# 02 · Recopilación de datos con web scraping (Wikipedia)
**Proyecto final – Capstone de Ciencia de Datos Aplicada (IBM)** · Autora: Jhoselyn Vera

**Objetivo:** extraer las tablas de lanzamientos del Falcon 9 de Wikipedia (versión archivada del curso, `oldid=1027686922`)
y convertirlas en un DataFrame con BeautifulSoup.

In [ ]:
import re
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)

STATIC_URL = ("https://en.wikipedia.org/w/index.php?"
              "title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922")
response = requests.get(STATIC_URL, headers={"User-Agent": "Mozilla/5.0 (capstone-ibm)"}, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.string)

## Funciones auxiliares para extraer cada columna

In [ ]:
def date_time(cell):
    return [t.strip() for t in list(cell.strings)][0:2]

def booster_version(cell):
    return "".join([b for i, b in enumerate(cell.strings) if i % 2 == 0][0:-1])

def landing_status(cell):
    return [s for s in cell.strings][0]

def get_mass(cell):
    mass = unicodedata.normalize("NFKD", cell.text).strip()
    if mass:
        return mass[0:mass.find("kg") + 2]
    return 0

## Recorrer las tablas y extraer los registros

In [ ]:
launch_dict = {k: [] for k in [
    "Flight No.", "Launch site", "Payload", "Payload mass", "Orbit", "Customer",
    "Launch outcome", "Version Booster", "Booster landing", "Date", "Time"]}

for table in soup.find_all("table", "wikitable plainrowheaders collapsible"):
    for rows in table.find_all("tr"):
        flag = False
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        cells = rows.find_all("td")
        if not flag or len(cells) < 9:
            continue
        dt = date_time(cells[0])
        bv = booster_version(cells[1]) or (cells[1].a.string if cells[1].a else "")
        launch_dict["Flight No."].append(flight_number)
        launch_dict["Date"].append(dt[0].strip(","))
        launch_dict["Time"].append(dt[1])
        launch_dict["Version Booster"].append(bv)
        launch_dict["Launch site"].append(cells[2].a.string)
        launch_dict["Payload"].append(cells[3].a.string)
        launch_dict["Payload mass"].append(get_mass(cells[4]))
        launch_dict["Orbit"].append(cells[5].a.string)
        launch_dict["Customer"].append(cells[6].a.string if cells[6].a else cells[6].text.strip())
        launch_dict["Launch outcome"].append(list(cells[7].strings)[0])
        launch_dict["Booster landing"].append(landing_status(cells[8]))

df = pd.DataFrame(launch_dict)
print("Registros extraídos:", len(df))
df.head()

In [ ]:
df.to_csv("../data/spacex_web_scraped.csv", index=False)
print("Guardado en data/spacex_web_scraped.csv")